**Our different attempts to extract the brfss data, these 3 are not used anymore, but for referencing history**

In [ ]:
# original brfss extraction, by col+manual

pct_medical_cost <- read.csv("out/medical_cost.csv") %>%
  select(PHR, group, pct, low, high, answer)
pct_medical_cost <- pct_medical_cost %>%
  filter(group == "Total" & answer == "YES")
pct_medical_cost <- pct_medical_cost[, !names(pct_medical_cost) %in% c("group", "answer", "low", "high")]
pct_medical_cost <- pct_medical_cost %>% rename(medical_cost_pct = pct)
print(pct_medical_cost)

past_flu_shot <- read.csv("out/flu_shot.csv") %>%
  select(PHR, group, pct, low, high, answer)
past_flu_shot <- past_flu_shot %>%
  filter(group == "Total" & answer == "YES")
past_flu_shot <- past_flu_shot[, !names(past_flu_shot) %in% c("group", "answer", "low", "high")]
past_flu_shot <- past_flu_shot %>% rename(flu_shot_pct = pct)
print(past_flu_shot)

brfss_data <- pct_medical_cost %>%
  inner_join(past_flu_shot, by = "PHR")
print(brfss_data)
write.csv(brfss_data, file = "data/brfss_data.csv", row.names = FALSE)

pneumonia_shot <- read.csv("out/pneumonia_shot.csv") %>%
  select(PHR, group, pct, low, high, answer)
pneumonia_shot <- pneumonia_shot %>%
  filter(group == "Total" & answer == "YES")
pneumonia_shot <- pneumonia_shot[, !names(pneumonia_shot) %in% c("group", "answer", "low", "high")]
pneumonia_shot <- pneumonia_shot %>% rename(pneumonia_shot_pct = pct)
print(pneumonia_shot)

brfss_data <- read.csv("data/brfss_data.csv")
print(brfss_data)
brfss_data <- brfss_data %>%
  inner_join(pneumonia_shot, by = "PHR")
print(brfss_data)
write.csv(brfss_data, file = "data/brfss_data.csv", row.names = FALSE)

routine_check <- read.csv("out/routine_check.csv") %>%
  select(PHR, group, pct, low, high, answer)
routine_check <- routine_check %>%
  filter(group == "Total" & answer == "Within the past year")
routine_check <- routine_check[, !names(routine_check) %in% c("group", "answer", "low", "high")]
routine_check <- routine_check %>% rename(routine_check_pct = pct)
print(routine_check)


In [ ]:
# Second try on brfss extraction, only having D10 response
from pathlib import Path
import pandas as pd

SHEET = "Routine Checkup"

def read_medical_cost_block(xlsx_path: Path) -> pd.DataFrame:
    # Excel row 9 is header -> header=8; keep rows 10–37 -> nrows=28
    df = pd.read_excel(
        xlsx_path,
        sheet_name=SHEET,
        header=8,
        nrows=28,
        engine="openpyxl"
    )

    cols = list(df.columns)
    df = df.rename(columns={
        cols[0]: "group",
        cols[1]: "category",
        cols[2]: "sample_size",
        cols[3]: "yes_pct",
        cols[4]: "yes_ci",
        cols[5]: "no_pct",
        cols[6]: "no_ci",
    })

    df["group"] = df["group"].ffill()
    return df[["group","category","sample_size","yes_pct","yes_ci","no_pct","no_ci"]]

def ci_to_low_high(ci_series: pd.Series) -> pd.DataFrame:
    tokens = ci_series.astype(str).str.extract(
        r"^\(\s*([0-9.]+|\.)\s*-\s*([0-9.]+|\.)\s*\)$"
    )
    low = pd.to_numeric(tokens[0].replace(".", pd.NA), errors="coerce")
    high = pd.to_numeric(tokens[1].replace(".", pd.NA), errors="coerce")
    return pd.DataFrame({"ci_low": low, "ci_high": high})

def to_long_yes_no(df: pd.DataFrame) -> pd.DataFrame:
    yes = pd.DataFrame({
        "group": df["group"],
        "category": df["category"],
        "sample_size": df["sample_size"],
        "response": "YES",
        "pct": pd.to_numeric(df["yes_pct"], errors="coerce"),
    }).join(ci_to_low_high(df["yes_ci"]))

    no = pd.DataFrame({
        "group": df["group"],
        "category": df["category"],
        "sample_size": df["sample_size"],
        "response": "NO",
        "pct": pd.to_numeric(df["no_pct"], errors="coerce"),
    }).join(ci_to_low_high(df["no_ci"]))

    return pd.concat([yes, no], ignore_index=True)

def process_one_file(xlsx_path: Path) -> pd.DataFrame:
    base = read_medical_cost_block(xlsx_path)
    long = to_long_yes_no(base)
    long.insert(0, "source_file", xlsx_path.name)
    return long

def process_11_at_once(input_dir: str, output_csv: str, pattern: str="*.xlsx") -> pd.DataFrame:
    in_dir = Path(input_dir)
    files = sorted([p for p in in_dir.glob(pattern) if p.suffix.lower()==".xlsx" and not p.name.startswith("~$")])
    if len(files) == 0:
        raise FileNotFoundError(f"No .xlsx files matched {pattern} in {in_dir}")

    all_long = pd.concat([process_one_file(p) for p in files], ignore_index=True)

    out_path = Path(output_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    all_long.to_csv(out_path, index=False)
    return all_long

# Example:
df_all = process_11_at_once(
    input_dir=r"data/brfss",
    output_csv=r"out/intermediate.csv",
    pattern="*.xlsx"
)
print(df_all.shape)

In [ ]:
"""
Kept a part of the answers
BRFSS Summary Tables Extractor
================================
Extracts Total + demographic breakdowns from all BRFSS PHR summary sheets
into a clean 3-sheet Excel workbook.

Usage:
    pip install openpyxl pandas
    python brfss_extract.py

Inputs / Outputs (edit the two paths below):
    SRC  – path to your source .xlsx file
    OUT  – path for the output .xlsx file
"""

import re
from openpyxl import load_workbook, Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── CONFIGURE THESE TWO PATHS ─────────────────────────────────────────────────
SRC = "data/raw/brfss/2024_PHR1_BRFSS_Summary_Tables.xlsx"   # <- your source file
OUT = "BRFSS_1Extracted.csv"                   # <- output file
# ─────────────────────────────────────────────────────────────────────────────

# Sheets to include in the "Disease Focus" tab.
# Edit this list to match your 11 (or any number of) target sheets.
DISEASE_SHEETS = [
    "Heart Disease", "Heart Attack", "Coronary Heart Disease", "CVD", "Stroke",
    "Diabetes", "COPD", "Arthritis", "Depressive Disorder", "Kidney Disease",
    "Current Asthma", "Skin Cancer", "Other Cancer", "Any Cancer",
    "Ever Told High BP", "Fair or Poor Health", "General Health",
    "Obese", "Overweight or Obese", "BMI 3 Categories",
    "Leisure Time PA", "Current Smoker", "Heavy Drinking", "Binge Drinking",
]

# For sheets where the FIRST response column is the "healthy/good" group,
# set the dict value to the 0-based index of the column you actually want.
# Format: "Sheet Name": column_index_of_percent  (3 = first %, 5 = second %, 7 = third %)
PRIMARY_COL_OVERRIDES = {
    # Example: these sheets list "None to <5 days" first; override to the 5+ column
    "Poor Physical Health 5+ Days":  5,
    "Poor Mental Health 5+ Days":    5,
    "Hlth Affected Activ 5+ Days":   5,
    "Poor Physical Health 14+ Day":  5,
    "Poor Mental Health 14+ Days":   5,
    "Hlth Affected Activ 14+ Days":  5,
}

# Sheets to skip entirely
SKIP_SHEETS = {"Index"}


# ── Styles ────────────────────────────────────────────────────────────────────
H1         = Font(name="Arial", bold=True, size=11, color="FFFFFF")
DT         = Font(name="Arial", size=10)
BD         = Font(name="Arial", bold=True, size=10)
FILL_BLUE  = PatternFill("solid", start_color="1F4E79")  # dark blue  – header
FILL_TOTAL = PatternFill("solid", start_color="D6E4F0")  # light blue – total rows
THIN       = Side(style="thin", color="BFBFBF")
BORDER     = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

def _sc(cell, font=None, fill=None, align="left", bold=False):
    if font: cell.font  = font
    if fill: cell.fill  = fill
    cell.alignment = Alignment(horizontal=align, vertical="center", wrap_text=False)
    cell.border    = BORDER

def _hdr(ws, columns, widths):
    """Write a styled header row and set column widths."""
    for c, h in enumerate(columns, 1):
        cell = ws.cell(row=1, column=c, value=h)
        _sc(cell, font=H1, fill=FILL_BLUE, align="center")
    ws.row_dimensions[1].height = 28
    for i, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = w
    ws.auto_filter.ref = f"A1:{get_column_letter(len(columns))}1"
    ws.freeze_panes    = ws.cell(row=2, column=1)


NOTE_PAT = re.compile(r"^(Note:|Software:|Prepared)", re.IGNORECASE)

def parse_sheet(ws, sheet_name):
    """
    Parse one BRFSS summary sheet and return a dict with:
        topic, age_grp, question, categories, primary_cat, primary_col, data
    Returns None if the sheet is too short to be a data sheet.
    """
    rows = list(ws.iter_rows(values_only=True))
    if len(rows) < 10:
        return None

    topic    = str(rows[2][0]).strip() if rows[2][0] else ""
    age_grp  = str(rows[3][0]).strip() if rows[3][0] else ""
    question = str(rows[4][0]).strip() if rows[4][0] else ""

    # Row 7 (0-indexed) = response category names
    # Row 8            = column sub-headers (Demographic Groups / Sample Size / % / 95% CI …)
    cat_row = rows[7]

    # Build list of (col_index, category_label) for every response category
    categories = []
    for i in range(3, len(cat_row), 2):
        v = cat_row[i]
        if v and str(v).strip() not in ("", "None"):
            categories.append((i, str(v).strip()))

    # Default: first category's % column
    default_col = categories[0][0] if categories else 3
    primary_col = PRIMARY_COL_OVERRIDES.get(sheet_name, default_col)

    # Find the matching category label for the chosen column
    primary_cat = next(
        (label for idx, label in categories if idx == primary_col),
        categories[0][1] if categories else ""
    )

    # Data rows start at index 9
    data_rows      = []
    current_group  = ""
    for r in rows[9:]:
        # Stop at blank separator or footnote rows
        if r[0] is None and r[1] is None:
            break
        if NOTE_PAT.match(str(r[0] or "")):
            break

        grp    = str(r[0]).strip() if r[0] else ""
        subgrp = str(r[1]).strip() if r[1] else ""
        if grp:
            current_group = grp

        n   = r[2]
        pct = r[primary_col]     if len(r) > primary_col     else None
        ci  = r[primary_col + 1] if len(r) > primary_col + 1 else None

        data_rows.append({
            "group":   current_group,
            "subgroup": subgrp,
            "n":        n,
            "pct":      pct,
            "ci":       ci,
        })

    return {
        "topic":       topic,
        "age_grp":     age_grp,
        "question":    question,
        "categories":  categories,
        "primary_cat": primary_cat,
        "primary_col": primary_col,
        "data":        data_rows,
    }


def build_output(parsed_all):
    wb = Workbook()

    # ── Sheet 1: Summary (one Total row per topic) ────────────────────────────
    ws1 = wb.active
    ws1.title = "Summary - Totals"
    _hdr(ws1,
         ["Sheet Name", "Topic / Category", "Age Group", "Question / Variable",
          "Primary Outcome", "Total N", "Total %", "95% CI"],
         [28, 48, 20, 60, 28, 10, 12, 20])

    for sheet_name, parsed in parsed_all.items():
        total = next((d for d in parsed["data"] if d["subgroup"] == "Total"), None)
        row   = [
            sheet_name,
            parsed["topic"],
            parsed["age_grp"],
            parsed["question"],
            parsed["primary_cat"],
            total["n"]   if total else "",
            total["pct"] if total else "",
            total["ci"]  if total else "",
        ]
        r = ws1.max_row + 1
        for c, val in enumerate(row, 1):
            cell = ws1.cell(row=r, column=c, value=val)
            _sc(cell, font=DT, fill=FILL_TOTAL)

    # ── Sheet 2: Full Detail ──────────────────────────────────────────────────
    ws2 = wb.create_sheet("Full Detail - All Demographics")
    _hdr(ws2,
         ["Sheet Name", "Topic / Category", "Primary Outcome",
          "Demographic Group", "Subgroup", "Sample N", "% (Primary)", "95% CI"],
         [28, 48, 28, 22, 30, 10, 14, 20])

    for sheet_name, parsed in parsed_all.items():
        for d in parsed["data"]:
            is_total = (d["subgroup"] == "Total")
            r        = ws2.max_row + 1
            row      = [
                sheet_name, parsed["topic"], parsed["primary_cat"],
                d["group"], d["subgroup"], d["n"], d["pct"], d["ci"],
            ]
            for c, val in enumerate(row, 1):
                cell = ws2.cell(row=r, column=c, value=val)
                _sc(cell,
                    font=BD if is_total else DT,
                    fill=FILL_TOTAL if is_total else None)

    # ── Sheet 3: Disease Focus ────────────────────────────────────────────────
    ws3 = wb.create_sheet("Disease Focus - Total + Demog")
    _hdr(ws3,
         ["Topic / Category", "Primary Outcome",
          "Demographic Group", "Subgroup", "Sample N", "% (Primary)", "95% CI"],
         [50, 28, 22, 30, 10, 14, 20])

    prev_topic = None
    for sheet_name in DISEASE_SHEETS:
        if sheet_name not in parsed_all:
            continue
        parsed = parsed_all[sheet_name]
        for d in parsed["data"]:
            is_total = (d["subgroup"] == "Total")
            is_new   = (parsed["topic"] != prev_topic)
            r        = ws3.max_row + 1
            row      = [
                parsed["topic"] if is_new else "",
                parsed["primary_cat"],
                d["group"], d["subgroup"],
                d["n"], d["pct"], d["ci"],
            ]
            for c, val in enumerate(row, 1):
                cell = ws3.cell(row=r, column=c, value=val)
                _sc(cell,
                    font=BD if is_total else DT,
                    fill=FILL_TOTAL if is_total else None)
            if is_new:
                prev_topic = parsed["topic"]
        ws3.append([""] * 7)   # blank row between topics

    return wb


def main():
    print(f"Reading: {SRC}")
    wb_in      = load_workbook(SRC, read_only=True)
    parsed_all = {}

    for sheet_name in wb_in.sheetnames:
        if sheet_name in SKIP_SHEETS:
            continue
        parsed = parse_sheet(wb_in[sheet_name], sheet_name)
        if parsed:
            parsed_all[sheet_name] = parsed

    print(f"  Parsed {len(parsed_all)} sheets")

    wb_out = build_output(parsed_all)
    wb_out.save(OUT)
    print(f"Saved:  {OUT}")
    print("Output sheets:", [s.title for s in wb_out.worksheets])


if __name__ == "__main__":
    main()

**2 versions of base inspection with the medical cost pct**
1. We first examine the trend by different categories through visualization
2. We do regression on the medical cost pct

In [ ]:
# visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

df = pd.read_csv("data/brfss/combined_with_phr.csv")

# --- filter PHR=1, YES ---
df1 = df.loc[(df["PHR"] == 1) & (df["answer"] == "YES")].copy()

# --- coerce numerics safely ---
for col in ["pct", "low", "high"]:
    df1[col] = pd.to_numeric(df1[col], errors="coerce")

df1["n"] = (
    df1["n"].astype(str)
    .str.replace(",", "", regex=False)
    .replace({"N": np.nan, "": np.nan, "nan": np.nan})
)
df1["n"] = pd.to_numeric(df1["n"], errors="coerce")

# Keep only rows usable for errorbars
df1 = df1.dropna(subset=["pct", "low", "high", "group", "level"])
df1 = df1[(df1["high"] >= df1["pct"]) & (df1["pct"] >= df1["low"])]

groups = sorted(df1["group"].unique())
if len(groups) == 0:
    raise ValueError("No groups found after filtering. Check PHR/answer filters and missing values.")

# --- layout (auto grid) ---
n_groups = len(groups)
ncols = 3  # change to 2 or 4 if you want
nrows = math.ceil(n_groups / ncols)

# Global x-limits for comparability
xmin = float(np.nanmin(df1["low"]))
xmax = float(np.nanmax(df1["high"]))

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(5.5 * ncols, 3.0 * nrows),
    sharex=True
)

# axes can be 2D array or 1D; flatten for easy indexing
axes = np.array(axes).reshape(-1)

for i, g in enumerate(groups):
    ax = axes[i]
    sub = df1[df1["group"] == g].sort_values("pct").copy()

    y = np.arange(len(sub))
    x = sub["pct"].to_numpy()
    xerr = np.vstack([(x - sub["low"].to_numpy()), (sub["high"].to_numpy() - x)])

    ax.errorbar(x, y, xerr=xerr, fmt="o", capsize=3, elinewidth=1, markersize=4)
    ax.set_yticks(y)
    ax.set_yticklabels(sub["level"].astype(str))
    ax.set_title(g)
    ax.grid(axis="x", linestyle="--", linewidth=0.5, alpha=0.5)
    ax.set_xlim(xmin, xmax)

# Hide any unused subplots
for j in range(n_groups, len(axes)):
    axes[j].axis("off")

fig.suptitle("PHR 1 — YES (%) by subgroup (with 95% CI)", y=1.02)
fig.supxlabel("YES (%)")
fig.tight_layout()
plt.show()

In [ ]:
# Regression
library(tidyverse)
library(dplyr)

medical_cost <- read.csv("out/medical_cost.csv")

# Official DSHS Public Health Region mapping (https://www.dshs.texas.gov/center-health-statistics/texas-county-numbers-public-health-regions) # nolint
tx_phr_lookup <- tribble(
~County, ~PHR, # nolint: indentation_linter.
"Anderson",4, # nolint: commas_linter, commas_linter.
"Andrews",9,
"Angelina",5,
"Aransas",11,
"Archer",2,
"Armstrong",1,
"Atascosa",8,
"Austin",6,
"Bailey",1,
"Bandera",8,
"Bastrop",7,
"Baylor",2,
"Bee",11,
"Bell",7,
"Bexar",8,
"Blanco",7,
"Borden",1,
"Bosque",7,
"Bowie",4,
"Brazoria",6,
"Brazos",7,
"Brewster",10,
"Briscoe",1,
"Brooks",11,
"Brown",2,
"Burleson",7,
"Burnet",7,
"Caldwell",7,
"Calhoun",8,
"Callahan",2,
"Cameron",11,
"Camp",4,
"Carson",1,
"Cass",4,
"Castro",1,
"Chambers",6,
"Cherokee",4,
"Childress",1,
"Clay",2,
"Cochran",1,
"Coke",9,
"Coleman",2,
"Collin",3,
"Collingsworth",1,
"Colorado",6,
"Comal",8,
"Comanche",2,
"Concho",9,
"Cooke",3,
"Coryell",7,
"Cottle",2,
"Crane",9,
"Crockett",9,
"Crosby",1,
"Culberson",10,
"Dallam",1,
"Dallas",3,
"Dawson",1,
"Deaf Smith",1,
"Delta",4,
"Denton",3,
"DeWitt",8,
"Dickens",1,
"Dimmit",8,
"Donley",1,
"Duval",11,
"Eastland",2,
"Ector",9,
"Edwards",8,
"Ellis",3,
"El Paso",10,
"Erath",3,
"Falls",7,
"Fannin",3,
"Fayette",7,
"Fisher",2,
"Floyd",1,
"Foard",2,
"Fort Bend",6,
"Franklin",4,
"Freestone",7,
"Frio",8,
"Gaines",1,
"Galveston",6,
"Garza",1,
"Gillespie",8,
"Glasscock",9,
"Goliad",8,
"Gonzales",8,
"Gray",1,
"Grayson",3,
"Gregg",4,
"Grimes",7,
"Guadalupe",8,
"Hale",1,
"Hall",1,
"Hamilton",7,
"Hansford",1,
"Hardeman",2,
"Hardin",5,
"Harris",6,
"Harrison",4,
"Hartley",1,
"Haskell",2,
"Hays",7,
"Hemphill",1,
"Henderson",4,
"Hidalgo",11,
"Hill",7,
"Hockley",1,
"Hood",3,
"Hopkins",4,
"Houston",5,
"Howard",9,
"Hudspeth",10,
"Hunt",3,
"Hutchinson",1,
"Irion",9,
"Jack",2,
"Jackson",8,
"Jasper",5,
"Jeff Davis",10,
"Jefferson",5,
"Jim Hogg",11,
"Jim Wells",11,
"Johnson",3,
"Jones",2,
"Karnes",8,
"Kaufman",3,
"Kendall",8,
"Kenedy",11,
"Kent",2,
"Kerr",8,
"Kimble",9,
"King",1,
"Kinney",8,
"Kleberg",11,
"Knox",2,
"Lamar",4,
"Lamb",1,
"Lampasas",7,
"La Salle",8,
"Lavaca",8,
"Lee",7,
"Leon",7,
"Liberty",6,
"Limestone",7,
"Lipscomb",1,
"Live Oak",11,
"Llano",7,
"Loving",9,
"Lubbock",1,
"Lynn",1,
"McCulloch",9,
"McLennan",7,
"McMullen",11,
"Madison",7,
"Marion",4,
"Martin",9,
"Mason",9,
"Matagorda",6,
"Maverick",8,
"Medina",8,
"Menard",9,
"Midland",9,
"Milam",7,
"Mills",7,
"Mitchell",2,
"Montague",2,
"Montgomery",6,
"Moore",1,
"Morris",4,
"Motley",1,
"Nacogdoches",5,
"Navarro",3,
"Newton",5,
"Nolan",2,
"Nueces",11,
"Ochiltree",1,
"Oldham",1,
"Orange",5,
"Palo Pinto",3,
"Panola",4,
"Parker",3,
"Parmer",1,
"Pecos",9,
"Polk",5,
"Potter",1,
"Presidio",10,
"Rains",4,
"Randall",1,
"Reagan",9,
"Real",8,
"Red River",4,
"Reeves",9,
"Refugio",11,
"Roberts",1,
"Robertson",7,
"Rockwall",3,
"Runnels",2,
"Rusk",4,
"Sabine",5,
"San Augustine",5,
"San Jacinto",5,
"San Patricio",11,
"San Saba",7,
"Schleicher",9,
"Scurry",2,
"Shackelford",2,
"Shelby",5,
"Sherman",1,
"Smith",4,
"Somervell",3,
"Starr",11,
"Stephens",2,
"Sterling",9,
"Stonewall",2,
"Sutton",9,
"Swisher",1,
"Tarrant",3,
"Taylor",2,
"Terrell",9,
"Terry",1,
"Throckmorton",2,
"Titus",4,
"Tom Green",9,
"Travis",7,
"Trinity",5,
"Tyler",5,
"Upshur",4,
"Upton",9,
"Uvalde",8,
"Val Verde",8,
"Van Zandt",4,
"Victoria",8,
"Walker",6,
"Waller",6,
"Ward",9,
"Washington",7,
"Webb",11,
"Wharton",6,
"Wheeler",1,
"Wichita",2,
"Wilbarger",2,
"Willacy",11,
"Williamson",7,
"Wilson",8,
"Winkler",9,
"Wise",3,
"Wood",4,
"Yoakum",1,
"Young",2,
"Zapata",11,
"Zavala",8
)

# create a multi-year summary of outbreak cases by PHR
library(tidyverse)
cases <- read_csv("/Users/zxt/Desktop/OR Lab 26/Measles-Outbreak-and-Public-Policy-Reluctance/2026SP-stat/data/count.csv")

cases <- cases %>%
  filter(County != "Texas")

cases_long <- cases %>%
  pivot_longer(
    cols = `2016`:`2025`,
    names_to = "Year",
    values_to = "Cases"
  ) %>%
  mutate(Year = as.integer(Year))

cases_long <- cases_long %>%
  left_join(tx_phr_lookup, by = "County")

phr_year_outbreaks <- cases_long %>%
  group_by(PHR, Year) %>%
  summarise(
    OutbreakCases = sum(Cases, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(PHR, Year)

phr_year_outbreaks

# Filter to 2025 outbreak totals
outbreak_2025 <- phr_year_outbreaks %>%
  filter(Year == 2025)


medical_cost_pct <- medical_cost %>%
  select(PHR, Yes_pct)

# Now merge
analysis_df <- outbreak_2025 %>%
  left_join(medical_cost_pct, by = "PHR")


# Run linear model
model <- lm(OutbreakCases ~ Yes_pct, data = analysis_df)

summary(model)

ggplot(analysis_df, aes(x = Yes_pct, y = OutbreakCases)) +
  geom_point(size = 3) +
  geom_smooth(method = "lm", se = TRUE) +
  labs(
    title = "2025 Outbreak Cases vs YES %",
    x = "YES %",
    y = "2025 Outbreak Case Count"
  ) +
  theme_minimal()

pmodel <- glm(OutbreakCases ~ Yes_pct, family = poisson(), data = analysis_df)

ggplot(analysis_df, aes(x = Yes_pct, y = OutbreakCases)) +
  geom_point(size = 3) +
  geom_smooth(method = "glm", se = TRUE) +
  labs(
    title = "2025 Outbreak Cases vs YES %",
    x = "YES %",
    y = "2025 Outbreak Case Count"
  ) +
  theme_minimal()

summary(pmodel)

# Export analysis_df to CSV
# write.csv(tx_phr_lookup, "/Users/zxt/Desktop/OR Lab 26/Measles-Outbreak-and-Public-Policy-Reluctance/2026SP-stat/data/brfss/analysis_results.csv", row.names = FALSE)
# print("Dataframe saved to analysis_results.csv")
